# Logistic Regression Activity: Loan Approval Prediction

## Your Mission: Can We Predict Who Gets Approved?

You are now working as a **data scientist for a bank**.

The bank receives many loan applications and wants to use historical application data to predict whether a new applicant is likely to be **approved**.

Your job is to:

1. Explore the data
2. Identify the target and features
3. Prepare the data
4. Split the data into training and testing sets
5. Train a Logistic Regression model
6. Make predictions and probabilities
7. Evaluate the model
8. Experiment with classification thresholds
9. Make a business recommendation



# Part 1 — Understand the Problem

### Business Scenario

A bank wants to make faster decisions about loan applications.

For each applicant, we have information such as:

- **Age** — applicant's age
- **Income** — annual income
- **CreditScore** — credit score
- **LoanAmount** — amount requested
- **LoanTerm** — requested loan term in months
- **Approved** — whether the loan was approved

Our target is:

- `0` → Rejected
- `1` → Approved

### Your main question

> **Can we use an applicant's information to predict whether their loan will be approved?**

Think about this before coding:

**What would make an applicant more likely to be approved?**

Write your hypothesis in your own words below.

- Applicants would likely be more approved if they had a high annual income combined with high credit score. Having both of these are signifiers that ensures that an applicant can pay back their loan in time. These information should also be adjacent to the loan amount and loan term. If the bank computes that an applicant's income is not enough to cover the loan amount in the given terms then there may be a possibility of rejection.

# Part 2 — Import the Libraries

You will need:

- `pandas` for working with the dataset
- `numpy` for numerical operations
- `matplotlib` for simple visualization
- `train_test_split` for splitting the data
- `LogisticRegression` for the model
- evaluation metrics from `sklearn.metrics`

### Your task

Complete the import statements.


In [486]:
# TODO: Import the required libraries

import pandas as pd
import numpy as np

# Add your imports below

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


# Part 3 — Generate the Dataset

To make this activity easy to run in Google Colab, the dataset is simulated locally.

This means you do **not** need to download a CSV file.

The simulation creates realistic-looking loan application records with patterns that a Logistic Regression model can learn.

### Run the cell below

You do not need to modify this cell.


In [487]:
# DO NOT MODIFY — dataset generation

np.random.seed(42)

n = 1500

age = np.random.randint(21, 66, n)
income = np.random.normal(55000, 18000, n).clip(18000, 150000)
credit_score = np.random.normal(680, 55, n).clip(500, 850)
loan_amount = np.random.normal(25000, 12000, n).clip(5000, 80000)
loan_term = np.random.choice([12, 24, 36, 48, 60], size=n)

# Create an approval probability using several applicant characteristics.
score = (
    0.035 * (credit_score - 650)
    + 0.000018 * (income - 50000)
    - 0.000035 * (loan_amount - 20000)
    - 0.015 * (loan_term - 36)
    + 0.015 * (age - 35)
)

probability = 1 / (1 + np.exp(-score))

approved = (np.random.random(n) < probability).astype(int)

df = pd.DataFrame({
    "Age": age,
    "Income": income.round(2),
    "CreditScore": credit_score.round(0).astype(int),
    "LoanAmount": loan_amount.round(2),
    "LoanTerm": loan_term,
    "Approved": approved
})

print("Dataset created successfully!")
print("Shape:", df.shape)

df.head()


Dataset created successfully!
Shape: (1500, 6)


,Age,Income,CreditScore,LoanAmount,LoanTerm,Approved
0,59,76007.85,693,41922.57,24,1
1,49,84179.95,756,23259.08,12,1
2,35,26740.71,625,21064.70,60,0
3,63,56331.33,665,34612.97,24,0
4,28,45684.39,619,11680.57,48,1


# Part 4 — Explore the Dataset 🔎

Before building a model, we need to understand our data.

Remember the lesson from the demo:

> **Do not start with the model. Start with the data.**

### Task 1

Find the number of rows and columns.

### Task 2

Display the first few rows.

### Task 3

Check the data types and missing values.

### Task 4

Look at the distribution of the target variable.


In [488]:
# TODO 1: Display the shape of the dataset
print(df.shape)

(1500, 6)


In [489]:
# TODO 2: Display the first 10 rows
df.head(10)

,Age,Income,CreditScore,LoanAmount,LoanTerm,Approved
0,59,76007.85,693,41922.57,24,1
1,49,84179.95,756,23259.08,12,1
2,35,26740.71,625,21064.70,60,0
3,63,56331.33,665,34612.97,24,0
4,28,45684.39,619,11680.57,48,1
5,41,44219.36,624,38764.74,48,0
6,59,64996.78,701,26713.71,60,1
7,39,43788.33,725,18066.99,24,1
8,43,46072.56,644,17749.99,48,0
9,31,51973.93,722,8398.45,60,1


In [490]:
# TODO 3: Inspect the dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          1500 non-null   int32  
 1   Income       1500 non-null   float64
 2   CreditScore  1500 non-null   int64  
 3   LoanAmount   1500 non-null   float64
 4   LoanTerm     1500 non-null   int64  
 5   Approved     1500 non-null   int64  
dtypes: float64(2), int32(1), int64(3)
memory usage: 64.6 KB


In [491]:
# TODO 4: Check for missing values
print(df.isnull().sum())

Age            0
Income         0
CreditScore    0
LoanAmount     0
LoanTerm       0
Approved       0
dtype: int64


In [492]:
# TODO 5: Check how many applicants were approved and rejected
print(df["Approved"].value_counts(), "\n")
print(df["Approved"].value_counts(normalize=True) * 100)

Approved
1    1001
0     499
Name: count, dtype: int64 

Approved
1    66.733333
0    33.266667
Name: proportion, dtype: float64


## Analyze Your Results

Answer these questions:

**1. How many applicants are in the dataset?**

- There are around 1500 applicants in the dataset.

**2. How many features do we have?**

- There are 5 features in the dataset which are age, income, credit score, loan amount, and loan term.

**3. What is our target variable?**

- The target variable is the "approved" status.

**4. How many applicants were approved?**

- There are around 1001 approved applicants or 66.73% of the population in the given dataset.

**5. How many were rejected?**

- There are around 499 rejected applicants or 33.27% of the population in the given dataset.

**6. Is the target reasonably balanced, or is one class much larger than the other?**

- The target is not reasonably balanced as there are more approved applicants than those who are not.

**7. Based on the target distribution, do you think accuracy alone will be enough to evaluate the model? Why?**

- In consideration of the unbalanced target distribution, accuracy will not be enough to evaluate the effectivity of the model as the disapproved group will be underrepresented.


# Part 5 — Identify X and y

We need to separate:

### Features (X)

The information used to make the prediction:

- Age
- Income
- CreditScore
- LoanAmount
- LoanTerm

### Target (y)

The outcome we want to predict:

- Approved

### Your task

Create `X` and `y`.


In [493]:
# TODO: Create X and y

X = df[["Age","Income","CreditScore","LoanAmount", "LoanTerm"]]

y = df["Approved"]

print("Features:")
print(X.head())
print("\nTarget:")
print(y.head())

Features:
   Age    Income  CreditScore  LoanAmount  LoanTerm
0   59  76007.85          693    41922.57        24
1   49  84179.95          756    23259.08        12
2   35  26740.71          625    21064.70        60
3   63  56331.33          665    34612.97        24
4   28  45684.39          619    11680.57        48

Target:
0    1
1    1
2    0
3    0
4    1
Name: Approved, dtype: int64


# Part 6 — Split the Data

We need two groups:

**Training data** → used to teach the model

**Testing data** → used to evaluate the model on unseen applicants

Use:

- 80% training
- 20% testing
- `random_state=42`
- `stratify=y` - meaning lang nito split yung data fairly as much as possible. lagay niyo lang after random state

### Your task

Write the train/test split code.


In [494]:
# TODO: Import train_test_split and split X and y

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# Part 7 — Train the Logistic Regression Model

Now we can build our classifier.

Remember from the demo:

> Logistic Regression learns patterns from the training data and produces a probability for each class.

### Your task

1. Import `LogisticRegression`
2. Create the model
3. Train it using `X_train` and `y_train`


In [495]:
# TODO: Train the Logistic Regression model

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

# Part 8 — Make Predictions

Now that the model has learned from the training data, ask it to predict the test applicants.

### Your task

Create a variable called `predictions`.

Expected values:

- `0` → Rejected
- `1` → Approved


In [496]:
# TODO: Make predictions for the test set

predictions = model.predict(X_test)

print(predictions[:20])

# Display the first 20 predictions

[0 0 0 1 1 1 1 1 1 1 1 1 0 0 1 0 1 1 1 1]


# Part 9 — Get Probabilities

This is one of the most important ideas.

A classification model does not only make a final `0` or `1` decision.

It can also tell us:

> **How confident is the model about the prediction?**

Use:

```python
model.predict_proba(X_test)
```

The second column represents the probability of the positive class:

**Approved = 1**

### Your task

Create `approval_probability`.


In [497]:
# TODO: Get the probability of approval

approval_probability = model.predict_proba(X_test)[:, 1]

# Display the first 10 probabilities

print(approval_probability[:10])


[0.44363366 0.1494499  0.47511321 0.9585001  0.75815239 0.91777734
 0.98786802 0.84843202 0.82984933 0.9429681 ]


### ✍️ Interpret One Applicant

After running your code, choose one applicant from the test set.

Record:

- Applicant's credit score
- Applicant's income
- Loan amount
- Actual approval status
- Predicted approval probability

Then explain:

> **What does the predicted probability mean?**

For example:

> "The model gives this applicant a 0.78 probability of approval. This means the model estimates a 78% chance of approval based on the patterns it learned from the training data. It does not guarantee that the bank should approve the applicant."


In [498]:
print("Customer information:")
print(X_test.iloc[0])
print("\nActual outcome:", y_test.iloc[0])
print("Predicted approval probability:", approval_probability[0])

Customer information:
Age               52.00
Income         79654.76
CreditScore      646.00
LoanAmount     37568.22
LoanTerm          60.00
Name: 1199, dtype: float64

Actual outcome: 1
Predicted approval probability: 0.44363365660556014


The model gives this applicant a 44% probability of approved despite the training data approving the applicant.

# Part 10 — The Classification Threshold

Our model gives probabilities, but the bank needs a decision:

**Approved or Rejected?**

The default threshold is usually:

### 0.50

So:

**Probability ≥ 0.50 → Approved**

**Probability < 0.50 → Rejected**

### Your task

Convert your probabilities into predictions using a threshold of 0.50.


In [499]:
# TODO: Convert approval probabilities into 0/1 predictions

threshold = 0.50

predictions_50 = (approval_probability >= threshold).astype(int)

print(predictions_50[:20])

[0 0 0 1 1 1 1 1 1 1 1 1 0 0 1 0 1 1 1 1]


# Part 11 — Evaluate the Model

Now we need to determine whether the model is useful.

Calculate:

- Accuracy
- Precision
- Recall
- F1-score

### Your task

Import the necessary metrics and calculate all four.

Do not just print the numbers.

Store them in variables so you can use them later.


In [500]:
# TODO: Import the evaluation metrics

In [501]:
# TODO: Calculate accuracy, precision, recall, and F1-score

accuracy = accuracy_score(y_test, predictions_50)
precision = precision_score(y_test, predictions_50)
recall = recall_score(y_test, predictions_50)
f1 = f1_score(y_test, predictions_50)

# Print your results
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1:", round(f1, 4))

Accuracy: 0.78
Precision: 0.8102
Recall: 0.875
F1: 0.8413


## ✍️ Interpret the Metrics

Complete the following:

### Accuracy
My model's accuracy is `0.78`

This means `the model is 78% in classifying applicants.`

### Precision
My model's precision for approval is `0.8102`

This means `the model is 81% accurate in terms of getting applicants approved correctly where the model can predict applicants that can pay properly to those who cannot.`

### Recall
My model's recall for approval is `0.875`

This means `the model is 86% accurate in terms of not getting potential applicants rejected when they are capable of repaying.`

### F1-score
My model's F1-score is `0.8413`

This tells me `the model has a solid balance between having high precision and having high recall.`


# Part 12 — Confusion Matrix

Now let's look at the **types of mistakes** the model makes.

The confusion matrix contains:

- **TN** — correctly rejected
- **TP** — correctly approved
- **FP** — predicted approved, but actually rejected
- **FN** — predicted rejected, but actually approved

### Your task

Create and display the confusion matrix.


In [502]:
# TODO: Create the confusion matrix

cm = confusion_matrix(
    y_test,
    predictions_50,
)

print(cm)

[[ 59  41]
 [ 25 175]]



### Question

> **Which mistake do you think is more costly for a bank: False Positive or False Negative?**

There is no automatic answer. **Defend your choice.**

* False positive is more costly which makes false negative  better for business to deny potential customers from being approved from those customers who are accidentally approved but cannot pay back their loans. If there are more false positives than false negatives then that means the bank would be approving loan for customers who actually cannot pay back their loans. Therefore, this would be way more costly than accidentally blocking a potential rich applicant.


# Part 13 — Classification Report

Use the classification report to see precision, recall, F1-score, and support for both classes.

### Your task

Generate the report.

Use:

- `Rejected`
- `Approved`

as the class names.


In [503]:
# TODO: Generate the classification report
print(classification_report(
    y_test,
    predictions_50,
    target_names=["Approved", "Not Approved"],
    zero_division=0
))


              precision    recall  f1-score   support

    Approved       0.70      0.59      0.64       100
Not Approved       0.81      0.88      0.84       200

    accuracy                           0.78       300
   macro avg       0.76      0.73      0.74       300
weighted avg       0.77      0.78      0.77       300



# Part 14 — Threshold Experiment 🔥

Now we will investigate one of the most important ideas from the demo:

> **The threshold can change the model's behavior.**

Test these thresholds:

```text
0.30
0.40
0.50
0.60
0.70
```

For every threshold, calculate:

- Accuracy
- Precision
- Recall
- F1-score

Store the results in a DataFrame.

### Your task

Complete the loop below.


In [504]:
# TODO: Complete the threshold experiment

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

results = []

for threshold in thresholds:

    # Convert probabilities into predictions
    predictions_threshold = (approval_probability >= threshold).astype(int)

    # Calculate the metrics
    accuracy_value = accuracy_score(y_test, predictions_threshold)
    precision_value = precision_score(y_test, predictions_threshold)
    recall_value = recall_score(y_test, predictions_threshold)
    f1_value = f1_score(y_test, predictions_threshold)

    # Add the results
    results.append({
        "Threshold": round(threshold, 4),
        "Accuracy": round(accuracy_value, 4),
        "Precision": round(precision_value, 4),
        "Recall": round(recall_value, 4),
        "F1": round(f1_value, 4)
    })

threshold_results = pd.DataFrame(results)

threshold_results


,Threshold,Accuracy,Precision,Recall,F1
0,0.3,0.7867,0.7656,0.980,0.8596
1,0.4,0.7833,0.7872,0.925,0.8506
2,0.5,0.7800,0.8102,0.875,0.8413
3,0.6,0.7500,0.8238,0.795,0.8092
4,0.7,0.7533,0.8663,0.745,0.8011


# Part 14 — The Bank's Decision

Now imagine you are presenting your model to the bank manager.

The manager tells you:

> **"Our biggest concern is approving applicants who are likely to default. We want to be careful about approving risky applicants."**

### Think about the threshold.

Would you prefer a:

**Lower threshold?**

or

**Higher threshold?**

### Your task

Recommend a threshold from the experiment:

> **I recommend a threshold of 0.7 because a higher threshold increases precision to its highest level (0.8663), which minimizes false positives. In this context, maximizing precision ensures the bank is extra strict and accurate when identifying risky applicants, drastically reducing the chances of mistakenly approving someone who is likely to default.**

Use **precision, recall, false positives, and false negatives** in your explanation where appropriate.


# Part 15 — Analyze the Threshold Experiment

Look at your completed table.

Answer:

### 1. Which threshold produced the highest accuracy?

`0.3 threshold produced the highest accuracy.`

### 2. Which threshold produced the highest precision?

`0.7 threshold produced the highest precision.`

### 3. Which threshold produced the highest recall?

`0.3 threshold produced the highest recall.`

### 4. Which threshold produced the highest F1-score?

`0.3 threshold produced the highest F1-score.`

### 5. What happens to recall as the threshold increases?

`The recall decreases as the threshold increases.`

### 6. What happens to precision as the threshold increases?

`The precision increases as the threshold increases.`

### 7. Why does changing the threshold change the predictions?

`A higher treshold makes the model much more strict and picky before making a positive claim.`

Use your results to support your explanation.


# Part 17 — Final Data Scientist Report

Your final answer should summarize the entire investigation.

## 1. Dataset

- Number of applicants: `1500`
- Number of features: `5`
- Target variable: `Approved`
- Class distribution: `1001 Approved : 499 Rejected (66.73% : 33.27%)`

## 2. Model

- Algorithm used: `Logistic Regression`
- Train/test split: `80/20`
- Features used: `Age Income, Credit Score, Loan Amount, & Loan Term`

## 3. Model Performance

- Accuracy: `0.78`
- Precision: `0.8102`
- Recall: `0.875`
- F1-score: `0.8413`

## 4. Confusion Matrix

Explain what your TP, TN, FP, and FN represent in this loan approval problem.

`TP (True Positive): The model correctly predicted 1 (Approved) for a safe applicant who can pay back the loan.`

`TN (True Negative): The model correctly predicted 0 (Rejected) for a risky applicant.`

`FP (False Positive): The model incorrectly predicted 1 (Approved) for a risky applicant who will default.`

`FN (False Negative): The model incorrectly predicted 0 (Rejected) for a safe applicant.`

## 5. Threshold Decision

- Recommended threshold: `0.7`
- Why?

`At a threshold of 0.7, the model achieves its highest precision (0.8663), which minimizes false positives. A strict threshold ensures the bank only flags an applicant as safe or risky when it has high confidence, directly addressing the manager's priority to avoid approving applicants who are likely to default.`

## 6. Business Recommendation

Answer:

> **Would you recommend that the bank use this model to support loan approval decisions? Why or why not?**

`Yes, but only as a decision-support tool alongside human review, not as a fully automated system as it is not nearly perfect.`

`With a threshold set to 0.7, the model delivers a high precision of 86.63%, effectively identifying high-risk applicants and helping the bank minimize costly default losses.`

`However, since the higher threshold trades off recall (dropping to 74.5%), the model experiences more false negatives (it misses ~25.5% of risky applicants). Furthermore, its overall accuracy is moderate (~75.33%).`

`Therefore, it serves as a valuable automated screening tool to flag obvious high-risk cases and streamline low-risk approvals, but human underwriters should review borderline applicants to catch missed defaults and prevent rejecting good customers.`